In [ ]:
# 1. What is a Vector Database (VectorDB) and how is it different from traditional databases?

- A Vector Database (VectorDB) is a specialized database designed to store, index, and search high-dimensional vector embeddings generated by machine learning models (such as text, image, or audio embeddings). These vectors represent semantic meaning, enabling similarity search and semantic retrieval.

Differences from Traditionally Databases:

. Traditional databases store structured data (rows,columns,keys) and use exact matching or range queries.
. VectorDBs store numerical vectors and perform approximate nearest neighbor (ANN) searches based on similarity metrics such as cosine similarity, Euclidean distance, or dot product.
. VectorDBs are optimized for AI use cases like semantic search, recommendation systems, clustering, and Retrieval-Augmented Generation (RAG).

# 2. Explain the various types of VectorDBs available and describe their suitability for different use cases.

- . Standalone Vector Databases (e.g., Pinecone, Weaviate, Milvus, chroma)
    - Designed purely for vector storage and similarity search.
    - Suitable for semantic search , recommendation systems, and RAG piplines.
  . Hybrid Databases (e.g., PostgreSQL + pgvector, MongoDB Atlas Vector Search)
    - Combine traditional relational/document storage with vector indexing.
    - Suitable when structured queries and vector search are both required.
  . In-Memory Vector Stores (e.g., FAISS)
    - High-performance libraries rather than full databases.
    - Suitable for research, prototyping, and offline similarity search.
  . Cloud-Managed Vector Services
    - Fully managed and scalable.
    - Suitable for production AI systems with large-scale embeddings and low - latency requirements.

# 3. Why is Chroma DB considered important in the context of AI/ML projects? Describe its key features.

- Chroma DB is important because it is lightweight, open-source, and designed specifically for embedding-based applications used in LLM piplines.

Key Features;
. Simple Python API and easy integration with Lnagchain and LlamaIndex.
. Persistent and in-memory storage options.
. Fast similarity search using embeddings.
. Metadata filtering and document storage.
. Ideal for RAG systems and semantic search in small to medium projects.

# 4. What are the benefitsnof using Hugging Face Hub for generative AI tasks?

- . Access to thousand of pre-trained models for NLp, Vision, and audio.
  . Open-source and community-driven ecosystem.
  . Easy model loading using the transformers libraray.
  . Version control, model cards, and documentation.
  . Supports fine-tining, deployment, and sharing models.

# 5. Describe the process and advantages of navigating and using pre-trained models from the Hugging Face Hub.

-  Process:
. Visit Hugging Face Hub and search by task(text generation, summarization, etc.).
. Review model cards for architecture, datasets, and usage.
. Load models using Automodel and AutoTokenizer in Python.
. Test inference and fine-tune if required.

- Advantages:
. Saves training time and cost.
. Provides state-of-the-art performance.
. Easy experimentation and reproducibility.
. Large community support and frequent updates.

In [ ]:
# 6. Install and set up Chroma DB, and insert sample vector data fro semantic search?

import chromadb
from sentence_transformers import SentenceTransformer
client = chromadb.Client()
collection = client.create_collection(name="demo_collection")
model = SentenceTransformer("all-MiniLM-L6-v2")
docs = [
    "Machine learning is a subfield of artificial intelligence that focuses on the development of algorithms and models that enable computers to learn and make predictions or decisions without being explicitly programmed.",
    "Deep learning uses neural networks",
    "Python is widely used for data science"
]
embeddings = model.encode(docs).tolist()
collection.add(
    documents=docs,
    embeddings=embeddings,
    ids=["1", "2", "3"]
)
query = "AI and neural networks"
query_embedding = model.encode(query).tolist()
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)
print(results)


In [ ]:
# 7. Demonstrate how to download and fine-tune a Hugging Face model for a text generation task.

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Fix: Add padding token
model = AutoModelForCausalLM.from_pretrained(model_name)
dataset = load_dataset("imdb",split="train[:1%]")

def tokenize(batch):
   return tokenizer(batch["text"], truncation=True,padding="max_length", max_length=128)

tokenized_data = dataset.map(tokenize, batched=True)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data
)

trainer.train()


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

In [ ]:
# 8. Create a custom LLm using Ollama and Llama2, and rune it locally for basic text prompts?

# Install and verify Ollama
!ollama --version

# Download Llama2 model
!ollama pull llama2

# Run Llama2
!ollama run llama2
# What is a vector database?

In [ ]:
# 9. Implement a basic RAG(Retrieveal-Augmented Generation) system using Ollama with Llama3.

!pip install chromadb sentence-transformers
import chromadb
from sentence_transformers import SentenceTransformer
import subprocess
import os
import time

# Ensure Ollama is in PATH for the current session
if '/usr/local/bin' not in os.environ['PATH']:
    os.environ['PATH'] += os.pathsep + '/usr/local/bin'

# Start Ollama server in the background if not already running
print("Checking and starting Ollama server...")
try:
    # Check if 'ollama serve' is already running using pgrep
    # This attempts to find an 'ollama serve' process
    subprocess.run(['pgrep', '-f', 'ollama serve'], check=True, capture_output=True)
    print("Ollama server is already running.")
except subprocess.CalledProcessError:
    print("Ollama server not found, starting it...")
    # Use Popen to start it in the background and detach from the current shell
    subprocess.Popen(
        ['ollama', 'serve'],
        stdout=subprocess.DEVNULL, # Redirect stdout to /dev/null
        stderr=subprocess.DEVNULL, # Redirect stderr to /dev/null
        preexec_fn=os.setsid # Detach process from current session group
    )
    # Give the server a moment to start up
    time.sleep(15) # Increased sleep time for better reliability in Colab
    print("Ollama server started (or attempted to start).")

# Pull Llama3 model if not already present
print("Checking and pulling llama3 model...")
try:
    # Check if the model is already available locally
    subprocess.run(['ollama', 'show', 'llama3'], check=True, capture_output=True, timeout=300)
    print("llama3 model already pulled.")
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    print("llama3 model not found or check timed out, pulling it now (this may take a while)...")
    try:
        pull_process = subprocess.run(['ollama', 'pull', 'llama3'], capture_output=True, text=True, check=True, timeout=1800) # 30 mins timeout for pull
        print(pull_process.stdout)
        print("llama3 model pulled successfully.")
    except subprocess.CalledProcessError as e:
        print(f"Error pulling llama3 model: {e.stderr}")
        raise # Re-raise to stop execution if model pull fails
    except subprocess.TimeoutExpired:
        print("Timeout while pulling llama3. Please check your internet connection or try again.")
        raise # Re-raise to stop execution if pull times out

# Setup vector DB
client = chromadb.Client()
# Use get_or_create_collection to avoid error if collection already exists
collection = client.get_or_create_collection(name="rag_demo")
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [
    "Diabetes is a chronic disease affecting blood sugar levels",
    "Hypertension is high blood pressure"
]

embeddings = model.encode(texts).tolist()
collection.add(documents=texts, embeddings=embeddings, ids=["1", "2"])

# Query
query = "What is diabetes?"
q_embed = model.encode([query]).tolist()
results = collection.query(query_embeddings=q_embed, n_results=1)
context = results['documents'][0][0]

# Generate answer with Ollama
prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

print(f"\nSending prompt to Ollama: {prompt}")
try:
    # Pass the prompt directly as an argument to ollama run for non-interactive mode
    ollama_run_result = subprocess.run(
        ["ollama", "run", "llama3", prompt],
        capture_output=True,
        text=True,
        check=True, # Raise CalledProcessError if the command returns a non-zero exit code
        timeout=600 # 10 minutes timeout for the LLM response generation
    )
    print("Ollama response:")
    print(ollama_run_result.stdout)
    if ollama_run_result.stderr:
        print("Ollama stderr (if any):")
        print(ollama_run_result.stderr)
except subprocess.CalledProcessError as e:
    print(f"Error running Ollama: Command exited with status {e.returncode}")
    print(f"Ollama stdout: {e.stdout}")
    print(f"Ollama stderr: {e.stderr}")
except subprocess.TimeoutExpired:
    print("Ollama run timed out. The model might be taking too long to respond.")
except FileNotFoundError:
    print("Ollama command not found. Ensure Ollama is installed and in PATH.")

In [ ]:
# 10. Health -Tech chatbot solution using Hugging Face, VectorDB, and Ollama.

# Encode medical documents using Hugging Face
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb")

# Store in VectorDB (Chroma)
# Retrieve relevant chunks
# Send context + question to Ollama Llama3 for final answer